In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns

## Other Discrete Distributions

You've already seen the Bernoulli distribution, with just two possible outcomes, heads or tails, True or False, 0 or 1.

More generally, a discrete distribution has discrete, or categorical outcomes. Generally, a finite number of possible outcomes. But technically, if we have a countably infinite number of possible outcomes, like all the integers, we still call it a discrete distribution.

Let's look at some other discrete distributions, with more than two possible outcomes (events). 

We'll follow the same approach of using a data generating function, the distribution, to generate a large sample; and then we'll plot the frequencies.

## Six-sided die

Let's start with a fair die, where each of the sides is equally weighted. Then we'll consider an unfair die.

In [ ]:
# Sample data from a discrete distribution with six possible labels, specifying the probability of each label
np.random.seed(42)


def sample_discrete(weights=[1, 1, 1, 1, 1, 1], n=1000, labels=None):
    if not labels:
        labels = range(1, len(weights) + 1)
    probs = weights / np.sum(weights)
    data = np.random.choice(labels, size=n, p=probs)
    df = pd.DataFrame(data, columns=["outcome"])
    df["outcome"] = pd.Categorical(df["outcome"], categories=labels, ordered=True)
    return df

In [ ]:
labels = ["one", "two", "three", "four", "five", "six"]
sample_discrete(n=20, labels=labels)

In [ ]:
sample_discrete(n=1000, labels=labels).outcome.value_counts().reindex(labels)

In [ ]:
def plot_discrete(data, ax1):
    # Create the bar plot on the first y-axis
    counts = data["outcome"].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=ax1)
    # Plot the normalized frequency on a second y-axis
    ax2 = ax1.twinx()
    freqs = data["outcome"].value_counts(normalize=True)
    sns.barplot(x=freqs.index, y=freqs.values, ax=ax2)


In [ ]:
plot_discrete(
    sample_discrete(
        weights=np.array([1, 1, 1, 1, 1, 1]),
        n=100000,
        labels=["one", "two", "three", "four", "five", "six"],
    ),
    plt.gca(),
)

What happens if we change the weights, trying to make our die come out with a lot more sixes?

In [ ]:
plot_discrete(
    sample_discrete(
        weights=np.array([1, 1, 1, 1, 1, 5]),
        n=100000,
        labels=["one", "two", "three", "four", "five", "six"],
    ),
    plt.gca(),
)

I can make a seven-sided die just as easily, with whatever weights I want

In [ ]:
plot_discrete(
    sample_discrete(weights=np.array([4, 1, 6, 2, 3, 4, 10]), n=100000), plt.gca()
)

## Poisson
The Poisson distribution is a discrete distribution that models the number of events occurring in a fixed interval of time or space.
- It is characterized by a single parameter, λ (lambda), which is the average number of events in each interval.
- The probability of observing k events in an interval is given by the Poisson probability mass function:
$
P(k;λ) = \frac{{λ^k * e^{-λ}}}{{k!}}
$

The Poisson distribution is used to model count data, such as the number of emails received in a day, the number of calls received in an hour, or the number of decay events per second from a radioactive source.

In [ ]:
# sample from a poisson distribution
def sample_poisson(lam=1, n=1000):
    data = np.random.poisson(lam=lam, size=n)
    return pd.DataFrame(data, columns=["outcome"])

In [ ]:
sample_poisson(lam=1, n=20)

In [ ]:
plot_discrete(sample_poisson(lam=1, n=100000), plt.gca())

In [ ]:
plot_discrete(sample_poisson(lam=2, n=100000), plt.gca())

In [ ]:
plot_discrete(sample_poisson(lam=10, n=100000), plt.gca())

In [ ]:
plot_discrete(sample_poisson(lam=1 / 10, n=100000), plt.gca())

## Zipf
What happens if we count up all the words in a large book? well, some words, like "the" are used extremely often. and some are rare, they only happen once. It turns out that when we plot the usage of words, it often follows something called a Zipf-distribution.

The zipf distribution has just one parameter, alpha, that controls the distribution of the frequencies.
A small alpha results in a distribution with many rare words, while a large alpha results in a distribution with fewer rare words.

The formula for the probability mass function of the zipf distribution is:

$$f(k; s, N) = \frac{1}{k^s} / H_{N,s}$$

where $s$ is the distribution parameter, $N$ is the number of elements, $H_{N,s}$ is the $N$-th generalized harmonic number, and $k$ is the rank of the element.

The generalized harmonic number is defined as:

$$H_{N,s} = \sum_{k=1}^{N} \frac{1}{k^s}$$


In [ ]:
def sample_zipf(alpha, n=1000):
    # use the scipy zipf distribution
    data = stats.zipf.rvs(a=alpha, size=n)
    return pd.DataFrame(data, columns=["outcome"])

In [ ]:
sample_zipf(alpha=1.5, n=20)

In [ ]:
# get counts
df = sample_zipf(alpha=1.5, n=10000)
df["outcome"].value_counts().sort_index()

In [ ]:
plot_discrete(df, plt.gca())

Uh-oh, Zipf's law! The distribution is highly skewed, with a few outcomes having very high frequencies. 
So, we can't see the majority of the data on the plot.

We're going to need to do a few things:
- Set a max-value on the y-axis that is much less than everything.
- Use a log scale on the y-axis. That is, instead of plotting the counts, we'll plot the log of the counts. 
- Use only a few tick marks on the x-axis
- relabel the x-axis as ranks, rather than outcomes

In [ ]:
def plot_zipf(data, ax1):
    # Create the bar plot on the first y-axis
    counts = data["outcome"].value_counts(ascending=False)
    counts.index = range(1, len(counts) + 1)
    # Use a log scale for the y-axis
    ax1.set_yscale("log")
    # Use a log scale for the x-axis
    sns.barplot(x=counts.index, y=counts, ax=ax1, order=counts.index)
    ticks = np.linspace(1, len(counts), 6, dtype=int)  # Create 6 evenly spaced tick positions
    ax1.set_xticks(ticks)  # Set the x-axis tick positions    
    # retitle the x-axis
    ax1.set_xlabel("Rank")


In [ ]:
plot_zipf(df, plt.gca())

In [ ]:
# make small multiple plots with increasing alpha
fig, axs = plt.subplots(3, 3, figsize=(12, 12))
for i, ax in enumerate(axs.ravel()):
    plot_zipf(sample_zipf(alpha=1.9-i/10, n=1000), ax)
    ax.set_title(f"alpha = {(1.9- i/10):.1f}")
plt.subplots_adjust(hspace=0.5)  # Add more space between the subplots

# 